[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/05_evaluation.ipynb)

# 04 - Evaluation

**Purpose.** Read the optimization runs back and answer three questions: what
changed against the deployed configuration, how the methods compare, and how
the coverage the KPIs measure relates to where the users actually are.

**Inputs.** The run directories under `outputs/optim/`, the baseline radio map
`data/interim/radio_map.npz`, and `data/processed/mdt.parquet`.

**Outputs.** Tables in `reports/tables/05_evaluation/` and figures in
`reports/figures/05_evaluation/`, so every number and picture here can be
looked at again without rerunning anything.

---

### This notebook needs no GPU

Unlike every other notebook in this project, nothing here re-solves. It reads
what the runs already wrote, so it imports neither Sionna-RT nor
`src.optim.evaluator`, and it finishes in seconds.

### Demand-weighted numbers are diagnostics, not objectives

Section 4 reports coverage weighted by where UEs stand, beside the tile-weighted
rates the KPIs use. Those two disagree sharply on this scenario. The
demand-weighted view is a **reporting** view: no optimizer has ever seen it, it
neither joins nor replaces the five KPIs of
[ADR 0001](../docs/adr/0001-five-kpis-under-lexicographic-priority.md), and it
cuts the map at the same `cfg.kpi` thresholds those KPIs do.


## 0. Environment

Run this section first, wherever you are.

**Locally** it walks up to the project root and makes it the working directory,
so the root-relative paths in the configs resolve the way they do for
`task bo` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path`, and installs
what Colab does not ship. No GPU runtime is needed here - but `outputs/` and
`data/` are not in the clone, so mount Drive below or there will be nothing to
evaluate.


In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook in this project.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). No sionna-rt: this notebook reads artifacts and
# never solves, so it runs on a CPU runtime. ax-platform is here only because
# src.optim.objective supplies the KPI vocabulary and the priority rule.
COLAB_PACKAGES = [
    ("hydra", "hydra-core"),
    ("ax", "ax-platform"),
]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR if SUBDIR else checkout
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ is DVC-tracked and not part of the clone, so a fresh Colab runtime has
# no scenario to optimize against. Either run notebook 00 first, or mount Drive
# and point the config at a copy that already holds one. Drive also survives a
# runtime reset, which /content does not.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# PROJECT_ROOT = "/content/drive/MyDrive/band-tilt"
# CONFIG_OVERRIDES += [
#     f"simulation.output.manifest_file={PROJECT_ROOT}/data/external/scenario.json",
#     f"data.output.mdt_file={PROJECT_ROOT}/data/processed/mdt.parquet",
#     f"optim.output.dir={PROJECT_ROOT}/outputs/optim",
# ]

## 1. Setup

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import json
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.evaluation import compare, maps, plots
from src.evaluation import runs as run_store
from src.evaluation.export import save_table
from src.utils.plotting import save_fig, setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()

# Every figure and table lands under reports/ so it can be read again without
# rerunning this notebook. Skipped on Colab, where /content does not survive.
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path("reports/figures/05_evaluation"))
save_table = partial(save_table, in_colab=IN_COLAB, directory=Path("reports/tables/05_evaluation"))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print(f"reading runs from {cfg.optim.output.dir}")

## 2. The runs, and whether they may be compared

`verify` refuses to put two maps on one axis unless they were solved the same
way. `README.md` names the ray-tracing settings, the grid resolution and the
KPI thresholds as the three things that invalidate stored results rather than
adding to them, so a difference on any of them would show up as a coverage
change that tilt did not cause.


In [ ]:
found = run_store.discover(cfg.optim.output.dir)
latest = run_store.latest_per_method(found)
runs = [latest[method] for method in sorted(latest)]
baseline = run_store.baseline_map(cfg)

checks = run_store.verify(runs, baseline)
run_store.require(checks)  # raises, naming every offender, rather than plotting nonsense

print(f"{len(found)} runs on disk, comparing the newest of each method:")
for run in runs:
    print(f"  {run.label}")
print(f"\n{checks['holds'].sum()}/{len(checks)} comparability checks hold")

### What the runs found

Read this before any figure. A budget too small to cover 36 dimensions will
honestly report that it found nothing, and no amount of downstream plotting
turns that into a result.


In [ ]:
print(compare.summarise(runs, cfg))

## 3. Before and after

The incumbent is the tilt table committed in `configs/simulation.yaml`, and its
radio map is the one the simulation stage already wrote - no run needs to store
it again.

The verdict column is the honest part. Each delta is compared against that
KPI's tolerance in `configs/kpi.yaml`, which sits at the ray tracer's own
run-to-run spread. Below it, the solver cannot tell the two configurations
apart, so the answer is a tie rather than a sign.


In [ ]:
mdt = pd.read_parquet(cfg.data.output.mdt_file)
cell_table = pd.read_parquet(cfg.data.output.cell_file)
cells = cell_table.drop_duplicates("cell")[["cell", "x", "y"]]
manifest = json.loads(Path(cfg.simulation.output.manifest_file).read_text(encoding="utf-8"))
hotspots = pd.DataFrame(manifest["density"]["hotspots"])

shape = maps.grid_shape(baseline)
counts = maps.demand(mdt, shape)
baseline_rsrp = baseline["rsrp_dbm"].astype(float)

winner = compare.best_method(runs, cfg)
deltas = {run.method: compare.delta_table(run.incumbent_kpi, run.best_kpi, cfg) for run in runs}

print(f"the priority order prefers {winner.label}\n")
save_table(deltas[winner.method], f"delta_{winner.method}")
deltas[winner.method]

In [ ]:
before = maps.best_server(baseline_rsrp)
after = maps.best_server(winner.radio_map["rsrp_dbm"].astype(float))

figure = plots.coverage_maps(before, after, baseline, cfg, cells=cells, label=winner.method)
save_fig(figure, "coverage_before_after")
plt.show()

## 4. Demand against signal

The five KPIs are computed over **tiles**, weighting every square of the map
equally. The users are not distributed that way, and on this scenario the
difference is large: the holes are mostly where nobody stands.

That does not make the hole rate wrong - it is the objective, and coverage of
empty ground still costs spectrum. It does mean a hole-rate improvement should
be read alongside how much of it a user would notice, which is what the CDF
below shows in one picture: the horizontal gap between the two curves at the
hole threshold.


In [ ]:
figure = plots.demand_signal_maps(
    baseline_rsrp, counts, baseline, cfg, cells=cells, hotspots=hotspots
)
save_fig(figure, "demand_vs_signal")
plt.show()

In [ ]:
figure = plots.coverage_cdf(before, counts, cfg)
save_fig(figure, "coverage_cdf")
plt.show()

In [ ]:
coverage = {"incumbent": maps.coverage_table(baseline_rsrp, counts, cfg)}
for run in runs:
    coverage[run.method] = maps.coverage_table(
        run.radio_map["rsrp_dbm"].astype(float), counts, cfg
    )

side_by_side = compare.coverage_comparison(coverage)
save_table(side_by_side, "coverage_by_area_and_demand")
side_by_side

## 5. Method against method

Matched on evaluations, not on wall clock. The model in `mobo` costs real time
that `random` does not spend, so a comparison reporting only the KPIs would
credit it for taking longer, and one reporting only the time would miss whether
it spent that time well. Both columns are here.


In [ ]:
table = compare.method_table(runs, cfg)
save_table(table, "method_comparison")
table

In [ ]:
figure = plots.kpi_comparison(deltas, cfg)
save_fig(figure, "kpi_improvement_vs_tolerance")
plt.show()

figure = plots.verdict_counts(deltas)
save_fig(figure, "verdict_per_kpi")
plt.show()

In [ ]:
trace = compare.convergence(runs)
save_table(trace, "convergence")

figure = plots.convergence_plot(trace)
save_fig(figure, "convergence")
plt.show()

In [ ]:
figure = plots.pareto_plot(runs, cfg)
save_fig(figure, "pareto_front")
plt.show()

## 6. How far the antennas moved

Reported, never optimized. A penalty on movement is an explicit non-goal, so
nothing in the objective has seen these numbers - they are here because a tilt
plan someone has to execute is easier to judge when the size of the change is
visible.


In [ ]:
movement = compare.tilt_movement(winner)
save_table(movement, f"tilt_movement_{winner.method}")
print(f"{winner.label}\n")
movement

In [ ]:
figure = plots.tilt_movement_plot(winner)
save_fig(figure, "tilt_movement")
plt.show()

save_table(winner.best_tilt, f"best_tilt_{winner.method}")

## 7. What was written

Every figure and table above was saved as it was produced, so this section only
lists the result. Nothing here needs rerunning to be read again.


In [ ]:
for directory in (Path("reports/figures/05_evaluation"), Path("reports/tables/05_evaluation")):
    written = sorted(directory.glob("*")) if directory.exists() else []
    print(f"{directory}: {len(written)} files")
    for path in written:
        print(f"  {path.name}")

## 8. Conclusions

State what was decided and on what evidence, not just the winning row.

- **Configuration to deploy:** `<method / run id>`
- **What it gained:** `<which KPIs moved past tolerance, and by how much>`
- **What it cost:** `<KPIs that worsened, and the wall clock>`
- **How a user would see it:** `<the demand-weighted change, not the tile one>`
- **Confidence:** `<were the budgets large enough for this to mean anything?>`
- **Reproduced by:** commit `<hash>`, scenario `<scenario_id>`


## 9. Handoff checklist

- [ ] Every run compared passed `verify` - same scenario, same fidelity, same thresholds
- [ ] The incumbent row matches `data/interim/radio_map.npz`
- [ ] Deltas smaller than their tolerance are read as ties, not as wins
- [ ] The demand-weighted numbers are reported as diagnostics, not as KPIs
- [ ] Budgets were large enough to compare methods, or the notebook says they were not
- [ ] Figures and tables are in `reports/`
